# Equipo 19 · Re-entrenar Endo: Endo-STTN reentrenado con C3VDv2 (Avance 3.1)

En el Avance 2, Endo-STTN se usa **tal como se publicó** (`gen_00009.pth`), entrenado con videos reales de
HyperKvasir. C3VDv2 son modelos de silicona: otra textura, otro color y otros reflejos. Aquí se **reentrena**
Endo-STTN con frames de C3VDv2 para que rellene mejor los reflejos de este dominio.

| Paso | Script (`ReEntrenar_Endo/`) | Qué hace |
|---|---|---|
| 1 | `preparar_endosttn.py` | Frames y **máscaras de reflejos** de C3VDv2 en el formato de Endo-STTN (288×288) |
| 2 | `entrenar_endosttn.py` | Entrenamiento original de Endo-STTN (generador + discriminador), partiendo de `gen_00009.pth` |
| 3 | `evaluar_endosttn.py` | Original vs reentrenado en videos no vistos, con *pseudo ground truth* |

**¿Hay que generar máscaras?** Sí, dos, y las dos salen solas:
1. **Máscara de reflejos** de cada frame: la misma del Avance 2 (píxeles casi blancos, `codigo/c3vd.py`). La genera el paso 1.
2. **Hueco desplazado** (*pseudo ground truth*): no hay frames de C3VDv2 sin reflejos para enseñarle a Endo-STTN el
   resultado correcto. Por eso la máscara de reflejos se **desplaza 50 px** sobre tejido sano, cuyo color sí se conoce.
   Endo-STTN aprende a rellenar ese hueco y se le mide el error ahí. Lo hace el propio código de Endo-STTN al
   entrenar (`core/dataset.py`) y `evaluar_endosttn.py` al evaluar.

La división de videos es la misma del Avance 2: Endo-STTN tampoco ve los videos de prueba.

## 0. Preparar Colab

Activa la GPU (*Entorno de ejecución → Cambiar tipo de entorno de ejecución*; mejor A100 o L4, T4 también sirve con `LOTE = 2`).
Usa los **mismos ZIPs de C3VDv2 de `MyDrive/Equipo19/raw/`** que el Avance 2: se leen directo de Drive, sin
copiarlos ni renombrarlos. Si existe `c3vd_preparado.zip` (lo guarda el notebook del Avance 2), de ahí toma la
división de videos; si no, la decide por nombre o con `DIVISION_MANUAL`. `gen_00009.pth` se descarga solo o se copia de `MyDrive/Equipo19/`.

Todo lo que produce se guarda directo en `MyDrive/Equipo19/re_entrenar_endo/`. **Si Colab se desconecta durante el
entrenamiento, vuelve a correr todas las celdas: continúa desde el último checkpoint.**

In [ ]:
%matplotlib inline
from pathlib import Path
import os, sys, json, shutil, subprocess, zipfile

# ---------- Ajusta si hace falta ----------
CORRER_PREPARAR = True
CORRER_ENTRENAR = True
CORRER_EVALUAR  = True
ITERACIONES  = 10000   # Endo-STTN original: 500 000 desde cero; aquí se parte de gen_00009.pth
GUARDAR_CADA = 1000    # un checkpoint cada N iteraciones (~66 MB cada uno)
LOTE = 4               # clips de 5 frames por lote; bájalo a 2 si falta memoria de GPU
PASO = 1               # 1 = todos los frames de cada video
# None = la división del Avance 2. Para elegirla a mano (nombres de los ZIPs, sin .zip):
# DIVISION_MANUAL = {"entrenamiento": ["video_1", "video_2"], "validacion": ["video_3"], "prueba": ["video_4"]}
DIVISION_MANUAL = None
DRIVE = Path("/content/drive/MyDrive/Equipo19")
REPO_URL = "https://github.com/Jkalks2025/Eliminaci-n-de-artefactos-en-im-genes-de-endoscop-a.git"
# ------------------------------------------

assert "google.colab" in sys.modules, "Este notebook es para Google Colab"
from google.colab import drive
drive.mount("/content/drive")
GUARDADO = DRIVE / "re_entrenar_endo"
GUARDADO.mkdir(parents=True, exist_ok=True)

REPO = Path("/content/repo")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown", "tabulate", "tensorboardX"], check=True)
pre = REPO / "Endo-STTN" / "release_model" / "pretrained_model"
if (DRIVE / "gen_00009.pth").exists() and not (pre / "gen_00009.pth").exists():
    pre.mkdir(parents=True, exist_ok=True); shutil.copy(DRIVE / "gen_00009.pth", pre)
subprocess.run([sys.executable, str(REPO / "preparar_entorno.py")], check=True)

datos = REPO / "datos"
assert (DRIVE / "raw").exists(), f"No encuentro {DRIVE / 'raw'}: ahí deben estar los ZIPs de C3VDv2"
for z in (DRIVE / "raw").glob("*.zip"):                    # ZIPs de C3VDv2: se leen directo de Drive, sin copiarlos
    if not (datos / "raw" / z.name).exists():
        os.symlink(z, datos / "raw" / z.name)
if (DRIVE / "c3vd_preparado.zip").exists() and not (datos / "c3vd" / "division.json").exists():
    with zipfile.ZipFile(DRIVE / "c3vd_preparado.zip") as z:   # solo la división del Avance 2
        if "c3vd/division.json" in z.namelist():
            z.extract("c3vd/division.json", datos)
if (GUARDADO / "endosttn_c3vd.zip").exists() and not (datos / "endosttn_c3vd" / "preparacion.json").exists():
    shutil.unpack_archive(str(GUARDADO / "endosttn_c3vd.zip"), str(datos))   # dataset ya preparado
(REPO / "modelos").mkdir(exist_ok=True)
(GUARDADO / "modelos").mkdir(exist_ok=True)
if not (REPO / "modelos" / "endosttn_c3vd").exists():     # checkpoints directo en Drive
    os.symlink(GUARDADO / "modelos", REPO / "modelos" / "endosttn_c3vd")
os.environ["EQUIPO19_RESULTADOS_31"] = str(GUARDADO / "resultados")
PESOS_ORIGINALES = pre / "gen_00009.pth"
print("Videos en Drive/raw:", sorted(z.stem for z in (DRIVE / "raw").glob("*.zip")))
print("División:", "DIVISION_MANUAL" if DIVISION_MANUAL else
      ("la del Avance 2 (c3vd_preparado.zip)" if (datos / "c3vd" / "division.json").exists() else "automática por nombre"))

In [ ]:
os.chdir(REPO)
A31 = REPO / "ReEntrenar_Endo"
sys.path.insert(0, str(A31)); sys.path.insert(0, str(REPO / "codigo"))
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from PIL import Image
from IPython.display import Markdown, display
import endosttn_c3vd as ec, limpiar_endosttn as le
from endosttn_c3vd import c3vd

def correr(*args):
    """Corre un script y muestra su salida mientras avanza."""
    p = subprocess.Popen([sys.executable, "-u", *map(str, args)], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, env=dict(os.environ, PYTHONWARNINGS="ignore"))
    for linea in p.stdout:
        print(linea, end="", flush=True)
    assert p.wait() == 0, f"falló: {args}"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COLOR = {"original": "#2a78d6", "reentrenado": "#eb6834"}
print("Dispositivo :", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "(sin GPU: entrenar será muy lento)")
print("ZIPs C3VDv2 :", sorted(z.stem for z in c3vd.DIR_RAW.glob("*.zip")))
print("Endo-STTN   :", PESOS_ORIGINALES, "OK" if PESOS_ORIGINALES.exists() else "NO ENCONTRADO")

## 1. Datos y máscaras

Cada frame pasa por lo mismo que en el Avance 2 (320×256 y máscara de reflejos por umbral de brillo) y después a
288×288, la resolución de Endo-STTN. Los videos de entrenamiento se guardan como ZIPs de frames (`JPEGImages/`) y de
máscaras (`Annotations/`) con los mismos nombres, que es el formato que lee Endo-STTN. Los de validación y prueba
se guardan aparte para evaluar.

In [ ]:
if CORRER_PREPARAR:
    extra = []
    if DIVISION_MANUAL:
        json.dump(DIVISION_MANUAL, open(GUARDADO / "division_manual.json", "w"), indent=2)
        extra = ["--division", GUARDADO / "division_manual.json"]
    correr(A31 / "preparar_endosttn.py", "--paso", PASO, *extra)
    if "drive" in globals():   # Colab: guardar el dataset preparado para la próxima vez
        shutil.make_archive(str(GUARDADO / "endosttn_c3vd"), "zip", root_dir=REPO / "datos", base_dir="endosttn_c3vd")

prep = json.load(open(ec.DIR_DATASET / "preparacion.json"))
display(pd.DataFrame(prep["videos"]).T)

In [ ]:
# Las dos máscaras en un frame de entrenamiento
trozo = sorted((ec.DIR_DATASET / "JPEGImages").glob("*.zip"))[0].stem
leer = lambda carpeta, i: np.array(Image.open(zipfile.ZipFile(ec.DIR_DATASET / carpeta / f"{trozo}.zip").open(f"{i:05d}.png")))
n = json.load(open(ec.DIR_DATASET / "train.json"))[trozo]
masks = [leer("Annotations", i) > 127 for i in range(n)]
i = int(np.argmax([m.sum() for m in masks]))
frame, mask = leer("JPEGImages", i), masks[i]
hueco = ec.hueco_pseudo(mask)
entrada = frame.copy(); entrada[hueco] = 0

fig, ax = plt.subplots(1, 4, figsize=(17, 4.6))
ax[0].imshow(frame); ax[0].set_title(f"{trozo} · frame {i}")
ax[1].imshow(frame); ax[1].imshow(np.ma.masked_where(~mask, mask), cmap="autumn", alpha=0.8)
ax[1].set_title(f"1. máscara de reflejos ({100 * mask.mean():.1f}%)")
ax[2].imshow(frame); ax[2].imshow(np.ma.masked_where(~hueco, hueco), cmap="cool", alpha=0.8)
ax[2].set_title(f"2. hueco desplazado {ec.DESPLAZAMIENTO} px →")
ax[3].imshow(entrada); ax[3].set_title("lo que ve Endo-STTN al entrenar")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()
print("Endo-STTN debe reconstruir el hueco (azul); su color real se conoce, así que se puede medir el error.")

## 2. Entrenamiento

Se usa el entrenamiento original de Endo-STTN (`Endo-STTN/core/trainer.py`) sin modificarlo:
- **Pérdidas:** L1 dentro del hueco, L1 en el resto de la imagen y pérdida adversarial (hinge, peso 0.01).
- **Optimizador:** Adam con lr 1e-4 y betas (0, 0.99).
- **Clips:** de 5 frames, a veces consecutivos y a veces al azar dentro del mismo video.

El generador arranca con `gen_00009.pth`. El discriminador arranca desde cero, porque el modelo publicado no lo trae.
Cada `GUARDAR_CADA` iteraciones se guarda `gen_XXXXX.pth`. Si el entrenamiento se interrumpe, al volver a correr esta
celda continúa desde el último checkpoint.

In [ ]:
if CORRER_ENTRENAR:
    correr(A31 / "entrenar_endosttn.py", "--pesos", PESOS_ORIGINALES, "--iteraciones", ITERACIONES,
           "--guardar-cada", GUARDAR_CADA, "--lote", LOTE)
print(sorted(p.name for p in ec.DIR_ENTRENAMIENTO.glob("gen_[0-9]*.pth")))

In [ ]:
# Curvas de pérdida (TensorBoard de Endo-STTN; se promedian cada 100 iteraciones)
try:
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
    fig, ax = plt.subplots(1, 3, figsize=(16, 3.6))
    for a, (sub, tag, titulo) in zip(ax, [("gen", "loss/hole_loss", "L1 en el hueco"),
                                          ("gen", "loss/valid_loss", "L1 fuera del hueco"),
                                          ("dis", "loss/dis_vid_fake", "discriminador (fake)")]):
        ea = EventAccumulator(str(ec.DIR_ENTRENAMIENTO / sub)); ea.Reload()
        ev = ea.Scalars(tag) if tag in ea.Tags()["scalars"] else []
        a.plot([e.step for e in ev], [e.value for e in ev], color=COLOR["reentrenado"], lw=2)
        a.set_title(titulo); a.set_xlabel("iteración"); a.grid(alpha=0.3)
    if ev: plt.tight_layout(); plt.show()
    else: plt.close(fig); print("Aún no hay puntos: Endo-STTN registra las pérdidas cada 100 iteraciones.")
except Exception as e:
    print("No se pudieron leer las curvas de TensorBoard:", e)

## 3. Evaluación: original vs reentrenado

Se mide en el hueco desplazado de videos que Endo-STTN no vio al entrenar: **MAE** (0–255, menor es mejor) y
**PSNR** (dB, mayor es mejor). Con la validación se elige el mejor checkpoint (`gen_mejor.pth`). Con la prueba se
compara contra el original, frame por frame.

In [ ]:
if CORRER_EVALUAR:
    correr(A31 / "evaluar_endosttn.py", "--pesos-originales", PESOS_ORIGINALES)
display(Markdown(open(ec.DIR_RESULTADOS / "resumen.md").read()))

In [ ]:
ruta_val = ec.DIR_RESULTADOS / "validacion_por_checkpoint.csv"
if ruta_val.exists():
    val = pd.read_csv(ruta_val)
    orig, ck = val[val.modelo == "original"].iloc[0], val[val.modelo != "original"]
    mejor = json.load(open(ec.DIR_ENTRENAMIENTO / "mejor.json"))
    fig, ax = plt.subplots(figsize=(8, 3.8))
    ax.plot(ck.iteracion, ck.mae, "-o", color=COLOR["reentrenado"], lw=2, ms=8, label="reentrenado (checkpoints)")
    ax.axhline(orig.mae, color=COLOR["original"], lw=2, ls="--", label="original (gen_00009)")
    ax.plot([mejor["iteracion"]], [ck[ck.modelo == mejor["checkpoint"]].mae.iloc[0]], "o", ms=14,
            mfc="none", mec="#333333", mew=2, label="elegido")
    ax.set_xlabel("iteración"); ax.set_ylabel("MAE en el hueco (validación)"); ax.grid(alpha=0.3); ax.legend()
    ax.set_title("Error del relleno en validación por checkpoint")
    plt.tight_layout(); plt.show()

In [ ]:
# Ejemplo visual en un video de prueba: hueco desplazado (se conoce la respuesta) y reflejos reales
div = json.load(open(ec.DIR_DATASET / "division.json"))
video = div["prueba"][0]
rgb, mask = np.load(ec.DIR_EVAL / video / "rgb.npy"), np.load(ec.DIR_EVAL / video / "mask.npy")
hueco = ec.hueco_pseudo(mask)
i = int(np.argmax(hueco.reshape(len(hueco), -1).sum(1)))
modelos = {"original": le.cargar(PESOS_ORIGINALES, DEVICE),
           "reentrenado": le.cargar(ec.DIR_ENTRENAMIENTO / "gen_mejor.pth", DEVICE)}
ini = max(0, i - 10)
seg = slice(ini, min(len(rgb), i + 11))
rell = {k: le.limpiar_video(m, rgb[seg], hueco[seg], DEVICE)[i - ini] for k, m in modelos.items()}
limp = {k: le.limpiar_video(m, rgb[seg], mask[seg], DEVICE)[i - ini] for k, m in modelos.items()}

def zoom(m, lado=96):
    ys, xs = np.nonzero(m[i]) if m[i].any() else (np.array([144]), np.array([144]))
    cy, cx = int(np.median(ys)), int(np.median(xs))
    y0, x0 = np.clip(cy - lado // 2, 0, 288 - lado), np.clip(cx - lado // 2, 0, 288 - lado)
    return lambda a: a[y0:y0 + lado, x0:x0 + lado]

zh, zm = zoom(hueco), zoom(mask)
paneles = [("hueco: real (respuesta)", zh(rgb[i])), ("hueco: original", zh(rell["original"])),
           ("hueco: reentrenado", zh(rell["reentrenado"])),
           ("reflejo real: frame", zm(rgb[i])), ("reflejo: original", zm(limp["original"])),
           ("reflejo: reentrenado", zm(limp["reentrenado"]))]
fig, ax = plt.subplots(1, 6, figsize=(18, 3.4))
for a, (t, img) in zip(ax, paneles):
    a.imshow(img); a.set_title(t, fontsize=10); a.axis("off")
plt.suptitle(f"{video} · frame {i} (zoom)", y=1.02); plt.tight_layout(); plt.show()
for k in modelos:
    m = ec.metricas_hueco(rell[k], rgb[i], hueco[i])
    print(f"{k:12s} MAE en el hueco de este frame: {m['mae']:.1f}")

## 4. Resultados y siguiente paso

Los resultados quedan en `MyDrive/Equipo19/re_entrenar_endo/resultados/` y el modelo elegido en
`MyDrive/Equipo19/re_entrenar_endo/modelos/gen_mejor.pth` (mismo formato que `gen_00009.pth`).

Para el reporte, sube a GitHub `resumen.md`, `validacion_por_checkpoint.csv` y `prueba_por_frame.csv`, y este
notebook ejecutado, a `ReEntrenar_Endo/resultados/`.

**Siguiente paso (Avance 3.2):** limpiar los frames del Avance 2 con `gen_mejor.pth`
(`python codigo/limpiar_endosttn.py --pesos .../gen_mejor.pth`) y ajustar el estimador de profundidad con ellos, para
ver si el mejor relleno también mejora la profundidad respecto al modelo D.

In [ ]:
mejor = json.load(open(ec.DIR_ENTRENAMIENTO / "mejor.json"))
print("Modelo elegido:", mejor, "->", ec.DIR_ENTRENAMIENTO / "gen_mejor.pth")
print("Resultados    :", sorted(p.name for p in ec.DIR_RESULTADOS.iterdir()), "en", ec.DIR_RESULTADOS)